# Geospatial Clustering for Food Delivery Optimization

**Zero-to-hero tutorial notebook** for delivery analytics with geospatial ML.

You will learn how to:
1. Load and clean noisy logistics data.
2. Engineer geospatial + temporal + operational features.
3. Detect outliers with multi-method consensus.
4. Cluster delivery patterns using six algorithms.
5. Evaluate cluster quality and convert outputs to business insights.

Dataset: `data/raw/train.csv` from Kaggle (`gauravmalik26/food-delivery-dataset`).


## 1) Why Geospatial Clustering Matters

Food delivery operations generate high-volume location traces. Clustering helps answer:

- Which zones produce dense order demand?
- Where do delivery routes become slow due to traffic and distance?
- How can we design service zones for better ETA and lower delivery cost?

This notebook focuses on **unsupervised pattern discovery**.


In [1]:
# 1. Imports and environment setup
from __future__ import annotations

import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Make notebook runnable from both project root and notebooks/ folder
if (Path.cwd() / "src").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise RuntimeError("Could not find project root with src/ directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering


In [2]:
# 2. Project module imports
from src.data_loader import load_and_clean_data
from src.features import build_features, select_features
from src.clustering import run_all
from src.evaluation import evaluate_all, elbow_curve
from src.outlier_detection import detect_outliers
from src.visualization import (
    plot_clusters_2d,
    plot_delivery_map,
    plot_elbow_curve,
    plot_silhouette,
    plot_feature_distributions,
    plot_interactive_map,
)
from src.pipeline import GeospatialClusteringPipeline


## 2) Load Raw Data

We start with raw CSV inspection before any transformations.


In [3]:
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
raw_df = pd.read_csv(RAW_PATH)

print("Raw shape:", raw_df.shape)
display(raw_df.head(3))
display(raw_df.describe(include="all").T.head(10))


Raw shape: (45593, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,45593,45593,0x4607,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery_person_ID,45593,1320,PUNERES01DEL01,67,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery_person_Age,45593,23,35,2262,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery_person_Ratings,45593,29,4.8,7148,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Restaurant_latitude,45593.0,NaN,NaN,NaN,17.017729,8.185109,-30.905562,12.933284,18.546947,22.728163,30.914057
Restaurant_longitude,45593.0,NaN,NaN,NaN,70.231332,22.883647,-88.366217,73.17,75.898497,78.044095,88.433452
Delivery_location_latitude,45593.0,NaN,NaN,NaN,17.465186,7.335122,0.01,12.988453,18.633934,22.785049,31.054057
Delivery_location_longitude,45593.0,NaN,NaN,NaN,70.845702,21.118812,0.01,73.28,76.002574,78.107044,88.563452
Order_Date,45593,44,15-03-2022,1192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time_Orderd,45593,177,NaN,1731,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3) Data Cleaning and Validation

`load_and_clean_data` applies:

- whitespace normalization
- date/time parsing
- numeric casting
- weather/time text cleaning
- coordinate sanity filtering (India bounds)


In [4]:
df = load_and_clean_data(str(RAW_PATH), validate=True)
report = df.attrs.get("validation_report", {})

print("Clean shape:", df.shape)
print("Validation report:")
print(report)

display(df.head(3))


Clean shape: (40007, 20)
Validation report:
{'n_raw_rows': 45593, 'steps': {'whitespace_stripped': True, 'order_date_bad': 0, 'time_taken_cleaned': True, 'dropped_missing_coords_or_age_ratings': 1908, 'dropped_out_of_bounds': 3678}, 'n_clean_rows': 40007}


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,2,Snack,motorcycle,0,No,Urban,24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26


In [5]:
# Quick data quality audit
null_share = (df.isna().mean() * 100).sort_values(ascending=False)
display(null_share.head(15).to_frame("missing_pct"))

city_dist = df["City"].value_counts(dropna=False)
display(city_dist.to_frame("count"))


,missing_pct
multiple_deliveries,2.109631
ID,0.000000
Delivery_person_ID,0.000000
City,0.000000
Festival,0.000000
Type_of_vehicle,0.000000
Type_of_order,0.000000
Vehicle_condition,0.000000
Road_traffic_density,0.000000
Weatherconditions,0.000000


,count
City,
Metropolitian,29956
Urban,8866
NaN,1047
Semi-Urban,138


## 4) Feature Engineering

Clustering quality depends heavily on feature design.

Engineered features include:
- delivery distance (haversine)
- temporal dimensions (hour/day/month/day-of-week)
- speed proxy (`distance / duration`)
- encoded traffic, city, festival indicators
- operational columns like vehicle condition and multi-deliveries


In [6]:
feat_df = build_features(df.copy())
X = select_features(feat_df)

print("Feature dataframe shape:", feat_df.shape)
print("Clustering matrix shape:", X.shape)
display(feat_df.head(3))


Feature dataframe shape: (40007, 31)
Clustering matrix shape: (40007, 16)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,pickup_distance_km,delivery_hour,delivery_day,delivery_month,delivery_dayofweek,duration_min,speed_kmph,festival_binary,traffic_code,city_code
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,...,0.0,11,19,3,5,15.0,12.10,0,3,2
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,...,0.0,19,25,3,4,5.0,242.20,0,4,3
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,...,0.0,8,19,3,5,15.0,6.21,0,1,2


In [7]:
# Feature snapshots for intuition
interesting_cols = [
    "delivery_distance_km",
    "duration_min",
    "speed_kmph",
    "traffic_code",
    "city_code",
    "festival_binary",
]

display(feat_df[interesting_cols].describe().T)


,count,mean,std,min,25%,50%,75%,max
delivery_distance_km,40007.0,9.716910,5.598584,1.465067,4.657655,9.193021,13.631449,20.969489
duration_min,40007.0,9.795911,4.260850,1.000000,5.000000,10.000000,15.000000,41.000000
speed_kmph,40007.0,83.538191,111.420096,4.700000,30.590000,55.330000,100.730000,1258.140000
traffic_code,40007.0,2.383808,1.248101,0.000000,1.000000,2.000000,4.000000,4.000000
city_code,40007.0,2.692979,0.613751,0.000000,2.000000,3.000000,3.000000,3.000000
festival_binary,40007.0,0.019572,0.138524,0.000000,0.000000,0.000000,0.000000,1.000000


## 5) Outlier Detection Before Clustering

Outliers distort centroid-based methods. We aggregate signals from:

- coordinate bounds
- IQR thresholds
- Mahalanobis distance
- Isolation Forest

Consensus rule: mark point as outlier if >= 2 methods agree.


In [8]:
consensus_report, individual_reports = detect_outliers(
    feat_df,
    feature_matrix=X,
    methods=None,
    min_consensus_votes=2,
)

print("Consensus outliers:", consensus_report.n_outliers)
for r in individual_reports:
    print(f"- {r.name}: {r.n_outliers}")

clean_mask = ~consensus_report.mask
feat_clean = feat_df.loc[clean_mask].reset_index(drop=True)
X_clean = X[clean_mask]

print("Rows after outlier removal:", X_clean.shape[0])


Consensus outliers: 875
- coordinate_bounds: 0
- iqr: 2338
- mahalanobis: 201
- isolation_forest: 2001
Rows after outlier removal: 39132


## 6) Clustering Algorithms

We compare six methods:

1. K-Means
2. MiniBatch K-Means
3. DBSCAN
4. HDBSCAN
5. Agglomerative Clustering
6. Gaussian Mixture Model

To keep runtime practical in notebook environments, we use optional sampling.


In [9]:
FAST_MODE = True
SAMPLE_SIZE = 12000
rng = np.random.default_rng(42)

if FAST_MODE and len(X_clean) > SAMPLE_SIZE:
    sample_idx = rng.choice(len(X_clean), size=SAMPLE_SIZE, replace=False)
    sample_idx = np.sort(sample_idx)
    X_work = X_clean[sample_idx]
    feat_work = feat_clean.iloc[sample_idx].reset_index(drop=True)
else:
    X_work = X_clean
    feat_work = feat_clean.copy()

print("Working matrix shape:", X_work.shape)


Working matrix shape: (12000, 16)


In [10]:
clustering_results = run_all(X_work)
eval_df = evaluate_all(X_work, clustering_results)

display(eval_df)


Fewer than 2 non-noise samples; silhouette = NaN


,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,params
0,minibatch_kmeans,5,0,0.391624,0.799612,20787.124921,"{'n_clusters': 5, 'batch_size': 1024, 'random_..."
1,kmeans,5,0,0.391542,0.794240,20898.327713,"{'n_clusters': 5, 'random_state': 42, 'n_init'..."
2,agglomerative,5,0,0.373629,0.807208,20204.857518,"{'n_clusters': 5, 'linkage': 'ward'}"
3,hdbscan,10,142,0.274826,1.426986,6065.142526,"{'min_cluster_size': 50, 'min_samples': 10}"
4,gmm,5,0,-0.026929,2.588449,1693.731058,"{'n_components': 5, 'random_state': 42}"
5,dbscan,0,12000,NaN,NaN,NaN,"{'eps': 0.5, 'min_samples': 10}"


In [11]:
best_algo = eval_df.iloc[0]["algorithm"]
best_labels = clustering_results[best_algo].labels

print("Best algorithm by silhouette:", best_algo)
print("Best silhouette:", eval_df.iloc[0]["silhouette"])


Best algorithm by silhouette: minibatch_kmeans
Best silhouette: 0.3916240086746788


## 7) Visual Diagnostics

We generate core diagnostics:

- Elbow curve (`k` selection intuition)
- 2D cluster projection
- Geographic cluster map
- Silhouette shape
- Feature distributions by cluster
- Interactive Folium map


In [12]:
elbow_df = elbow_curve(X_work, k_range=range(2, 10))
elbow_path = plot_elbow_curve(elbow_df, filename="nb_elbow_curve.png")

# Use first two dimensions for quick 2D projection
cluster_plot_path = plot_clusters_2d(
    X_work[:, :2],
    best_labels,
    title=f"Notebook Clusters ({best_algo})",
    filename="nb_clusters_2d.png",
)

delivery_map_path = plot_delivery_map(
    feat_work,
    best_labels,
    title=f"Delivery Map ({best_algo})",
    filename="nb_delivery_map.png",
)

sil_path = plot_silhouette(
    X_work,
    best_labels,
    filename="nb_silhouette.png",
)

dist_path = plot_feature_distributions(
    feat_work,
    best_labels,
    features=["delivery_distance_km", "speed_kmph"],
    filename="nb_feature_distributions.png",
)

print("Saved plots:")
for p in [elbow_path, cluster_plot_path, delivery_map_path, sil_path, dist_path]:
    print("-", p)


Saved plots:
- /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_elbow_curve.png
- /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_clusters_2d.png
- /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_delivery_map.png
- /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_silhouette.png
- /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_feature_distributions.png


In [13]:
# Interactive map export
interactive_map_path = plot_interactive_map(
    feat_work,
    best_labels,
    title=f"Interactive Delivery Clusters ({best_algo})",
    filename="nb_interactive_map.html",
)

print("Interactive map saved to:", interactive_map_path)


Interactive map saved to: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_interactive_map.html


## 8) Business Insight Layer

Raw cluster IDs are not business value until translated.
We aggregate key operational signals by cluster.


In [14]:
analysis = feat_work.copy()
analysis["cluster"] = best_labels

cluster_kpis = (
    analysis.groupby("cluster", as_index=False)
    .agg(
        orders=("cluster", "count"),
        avg_distance_km=("delivery_distance_km", "mean"),
        avg_duration_min=("duration_min", "mean"),
        avg_speed_kmph=("speed_kmph", "mean"),
        avg_traffic_code=("traffic_code", "mean"),
        avg_vehicle_condition=("Vehicle_condition", "mean"),
    )
    .sort_values("orders", ascending=False)
)

display(cluster_kpis)


,cluster,orders,avg_distance_km,avg_duration_min,avg_speed_kmph,avg_traffic_code,avg_vehicle_condition
0,0,4856,4.918288,11.906507,25.157168,2.119234,1.029242
2,2,3872,11.843668,11.000517,64.856785,2.613378,1.010589
4,4,1768,13.005887,7.036199,110.871538,2.599548,0.983032
3,3,756,18.044634,4.883598,225.680860,2.534392,1.042328
1,1,748,13.003066,5.000000,156.036872,2.609626,1.033422


### Interpreting clusters for operations

Use this decision template:

- **High orders + low distance + high speed**: candidate for SLA tightening and batching.
- **High orders + high traffic + low speed**: candidate for rider rebalancing and surge planning.
- **Low orders + long distance**: candidate for dynamic pricing or service-radius redesign.

Extend this notebook by joining financial metrics (cost per km, cancellation, tip rates) for full unit economics.


## 9) One-call Pipeline Demo

If you want single-command execution with reports and plots, use pipeline wrapper.


In [15]:
demo_pipeline = GeospatialClusteringPipeline(
    data_path=str(RAW_PATH),
    algorithms=["kmeans", "minibatch_kmeans", "dbscan"],
    remove_outliers=True,
)

demo_report = demo_pipeline.run()
print("Pipeline best algorithm:", demo_report.best_algorithm)
print("Pipeline output dir:", demo_report.output_dir)
display(pd.DataFrame(demo_report.algorithm_results).T)


Fewer than 2 non-noise samples; silhouette = NaN


Pipeline best algorithm: kmeans
Pipeline output dir: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs


,n_clusters,n_noise,metrics
kmeans,5,0,"{'silhouette': 0.3453192843561955, 'davies_bou..."
minibatch_kmeans,5,0,"{'silhouette': 0.3424702136462056, 'davies_bou..."
dbscan,0,39132,"{'silhouette': nan, 'davies_bouldin': nan, 'ca..."


## 10) Next Steps

1. Tune algorithm-specific hyperparameters by city segment.
2. Add demand forecasting per cluster.
3. Add route-time simulation and dispatch optimization.
4. Deploy real-time scoring with geofencing.

Notebook complete.
